[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [NumPy, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/numpy-deep-dive.html)

# Random Numbers


## What you will be able to do

Make random numbers with a seeded `Generator` that gives the same results on every run, add noise to
the readings, sample hours, shuffle and split days, and estimate a probability by simulation.

## The idea

### The problem

Some calculations need numbers that nobody chose. Testing a quality check on the readings needs
readings with noise added, as a real sensor's would have. Checking 500 of Bergen's 8,760 hours by
hand needs 500 hours picked without favoring any part of the year. Setting aside a fifth of the days
to test a model on needs the days in a shuffled order. And a question such as "how likely is a week
at Bergen to include an hour below freezing" can be answered by trying 10,000 random weeks.

Each of those has to come out the same on every run. A notebook whose printed numbers change every
time it runs cannot be checked against its own prose, a colleague cannot reproduce its results, and a
bug that appeared on one run may never appear again.

### What a random number generator is

> A **random number generator** is an object that produces a stream of numbers that behave like
> random ones. In NumPy it is a `Generator`, made with `np.random.default_rng(seed)`. The seed fixes
> the stream, so two generators made with the same seed produce the same numbers in the same order,
> and every method call, such as `integers`, `normal` or `choice`, takes the next numbers from it.

### Why it works that way

The numbers come from an algorithm, PCG64 by default, which turns the seed into a starting state and
each state into the next. They only look random, which is why they are called pseudo-random, and why
a seed reproduces them exactly. Every call moves the state on, so calling `integers` twice on one
generator gives two different sets of numbers, while a new generator made with the same seed starts
the stream again. Without a seed, `default_rng()` takes one from the operating system, and every run
differs.

Older code calls `np.random.seed(2025)` and then functions such as `np.random.randint`, which all
share one hidden generator for the whole program. Any library that draws from it moves it on, so code
far from yours can change your numbers. NumPy's documentation recommends a `Generator` for new code,
and this notebook uses no other kind, apart from one Common error that shows the two do not mix.

### Where this shows up

The **Train and Test** notebook in the **scikit-learn, Deep Dive** guide splits a dataset into
training and test rows with a seed, which that library calls `random_state`. Simulations, and the
bootstrap, which measures uncertainty by sampling a dataset again and again, are built from `choice`
and `integers`. The random positions this notebook draws are fancy indexes, from the **Fancy
Indexing** notebook, and the 10,000 weeks of its simulation come from broadcasting.

### What this notebook covers

- a generator made from a seed, and the same numbers again from the same seed
- `integers`, `random` and `uniform`
- `normal`, to add noise to the readings
- `choice`, to sample hours without picking any twice
- `permutation`, to shuffle the days and split them
- a simulation that estimates a share from 10,000 random weeks

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
import numpy as np

rng = np.random.default_rng(2025)
print(rng.integers(1, 7, size=5))
print(rng.integers(1, 7, size=5))
print(np.random.default_rng(2025).integers(1, 7, size=5))
```

```
[3 6 6 3 6]
[5 4 6 5 6]
[3 6 6 3 6]
```

Five rolls of a die came first, then five different rolls from the same generator. A new generator
with the same seed started the stream again and rolled the first five again.

## Setup

Three imports, the year of readings, and a helper that finds an hour's position.

- `numpy`, imported as `np`, is the library, and the cell prints its version
- `math`, `datetime` and `timedelta` build the readings, and `hour_of_year` finds positions

`readings_list` holds four lists of 8,760 readings, one for each station in `STATIONS`, made by
`year_of_readings`, the formula the **Why NumPy** notebook explained, and `readings` is the same
values as one array of shape (4, 8760), with `nan` for Svalbard's silent day. `hour_of_year(7, 1, 12)`
gives the position of noon on 1 July along a station's row. Every cell that draws random numbers
makes its own generator with a fixed seed, so it prints the same values however often it runs.

In [1]:
import math
from datetime import datetime, timedelta

import numpy as np

STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
START = datetime(2025, 1, 1)
HOURS = 365 * 24


def year_of_readings(station):
    """A reading for every hour of 2025 at one station, as the sqlite3 guide makes them.
    Svalbard was silent on 2 March, so those 24 hours are None."""
    i = list(STATIONS).index(station)
    row = []
    for n in range(HOURS):
        hour = START + timedelta(hours=n)
        if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
            row.append(None)
        else:
            season = -math.cos(2 * math.pi * (n - 400) / HOURS)
            day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
            wobble = ((n * 37 + i * 101) % 17 - 8) / 10
            row.append(round(STATIONS[station] + 9 * season + 3 * day + wobble, 1) + 0.0)
    return row


readings_list = [year_of_readings(station) for station in STATIONS]
readings = np.array(readings_list, dtype=float)       # None becomes nan


def hour_of_year(month, day, hour=0):
    """The position along a station's row of an hour of 2025. hour_of_year(7, 1, 12) is noon on 1 July."""
    return (datetime(2025, month, day, hour) - START) // timedelta(hours=1)


print("numpy", np.__version__, "|", len(readings_list), "stations,", len(readings_list[0]), "hours each |", readings.shape)

numpy 2.1.3 | 4 stations, 8760 hours each | (4, 8760)


## Worked examples

### A generator from a seed

In [2]:
rng = np.random.default_rng(2025)

print(type(rng).__name__, type(rng.bit_generator).__name__)
print(rng.integers(1, 7, size=5))
print(rng.integers(1, 7, size=5))
print(np.random.default_rng(2025).integers(1, 7, size=5))

Generator PCG64
[3 6 6 3 6]
[5 4 6 5 6]
[3 6 6 3 6]


`rng` is a `Generator` driven by PCG64. Its second call drew new numbers, and a second generator with
the seed 2025 began with the same five as the first.

### integers, random and uniform

In [3]:
rng = np.random.default_rng(1)

print(rng.integers(0, 10, size=8))
print(rng.random(3).round(3))
print(rng.uniform(-0.5, 0.5, size=3).round(3))

[4 5 7 9 0 1 8 9]
[0.312 0.423 0.828]
[-0.091  0.05  -0.472]


`integers(low, high)` draws whole numbers from `low` up to but not including `high`, as `range` does,
which one of the Common errors below runs into. `random` draws decimals from 0 up to 1, and `uniform`
from any range you give it.

### Noise from normal

`normal(mean, standard deviation, size)` draws from the bell-shaped normal distribution. With
`size=readings.shape` there is one draw for every reading:

In [4]:
rng = np.random.default_rng(42)
noise = rng.normal(0.0, 0.3, size=readings.shape)
noisy = readings + noise

print(noise.shape)
print(noise.mean().round(3), noise.std().round(3))
print(readings[0, :3], noisy[0, :3].round(2))

(4, 8760)
0.002 0.301
[-3.6 -3.7 -3.7] [-3.51 -4.01 -3.47]


The 35,040 draws have a mean close to 0 and a standard deviation close to 0.3, as asked. `noisy` is
the readings as a sensor with that much error might report them, and a quality check meant to cope
with noise can be tested on it.

### A sample of hours with choice

`choice(HOURS, size=5, replace=False)` picks five of the positions 0 to 8,759, never the same one
twice. The positions are a fancy index, so they pick Bergen's readings at those hours:

In [5]:
rng = np.random.default_rng(7)
sample = rng.choice(HOURS, size=5, replace=False)

print(sample)
print(readings[0, sample])
for hour in sample:
    print(START + timedelta(hours=int(hour)))

[5473 5992 8273 5065 7858]
[13.9 16.6  4.2 14.8  3.8]
2025-08-17 01:00:00
2025-09-07 16:00:00
2025-12-11 17:00:00
2025-07-31 01:00:00
2025-11-24 10:00:00


A larger sample from the same generator gives hours to check by hand:

In [6]:
check = rng.choice(HOURS, size=500, replace=False)

print(np.unique(check).size)
print(readings[0, check].mean().round(2), readings[0].mean().round(2))

500
8.09 8.0


The 500 hours are all different, as `np.unique` confirms, and their mean of 8.09 is close to the
year's 8.0.

### Shuffling and splitting with permutation

`permutation(365)` returns the numbers 0 to 364 in a shuffled order. The first 292 of them and the
last 73 split Bergen's days four to one:

In [7]:
rng = np.random.default_rng(11)
bergen_days = readings[0].reshape(365, 24).mean(axis=1)
order = rng.permutation(365)
train, test = order[:292], order[292:]

print(order[:6])
print(train.size, test.size, np.intersect1d(train, test).size)
print(bergen_days[train].mean().round(2), bergen_days[test].mean().round(2))

[359 350 192  69 363 168]
292 73 0
8.24 7.04


`np.intersect1d` finds the days in both sets, and there are none. The 73 test days have a mean of
7.04, about a degree below the training days' 8.24, a gap that comes from which days the shuffle
happened to put where. A model tested on those days is tested on a slightly colder year.

### A simulation with 10,000 random weeks

Each random start is the first hour of a week of 168 hours. A column of starts broadcast against a
row of 168 offsets gives every hour of every week, and one fancy index reads them all:

In [8]:
rng = np.random.default_rng(2025)
starts = rng.integers(0, HOURS - 167, size=10_000)
weeks = starts[:, np.newaxis] + np.arange(168)
has_frost = (readings[0][weeks] < 0).any(axis=1)

print(weeks.shape)
print(has_frost.mean())

(10000, 168)
0.3291


`any(axis=1)` asked of each week whether it had an hour below freezing, and 32.9% of the random weeks
did. Every week that fits in the year can be tried too:

In [9]:
every_start = np.arange(HOURS - 167)
every_week = every_start[:, np.newaxis] + np.arange(168)

print(every_week.shape)
print(round((readings[0][every_week] < 0).any(axis=1).mean(), 4))

(8593, 168)
0.3352


All 8,593 weeks give 33.5%, so the estimate from 10,000 random ones was within a percentage point.
Simulation is for questions where trying every case would take too long, and here both could run,
which is what lets the estimate be checked.

## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/numpy-deep-dive/12-random-numbers-solutions.ipynb).

**1.** Make a generator with seed 7, roll ten dice with `integers`, and print the rolls and their mean.

In [10]:
# your code here

**2.** With seed 8, draw 1,000 values from a normal distribution with mean 8 and standard deviation 6.73, and print their mean and standard deviation, rounded to two places.

In [11]:
# your code here

**3.** With seed 9, pick three of the four station names at random, without replacement, using `choice`.

In [12]:
# your code here

**4.** With seed 10, shuffle the positions of Oslo's 365 daily means with `permutation`, and print the mean of the first 73 shuffled days and the mean of all 365, rounded to two places.

In [13]:
# your code here

**5.** With seed 12 and 10,000 random starting hours, estimate how often a 24-hour window at Tromso, row 3, is entirely below freezing, and compare it with the share among every window that fits in the year.

In [14]:
# your code here

## Common errors

### AttributeError: 'numpy.random._generator.Generator' object has no attribute 'randint'

In [15]:
rng = np.random.default_rng(2025)
rng.randint(1, 7)

AttributeError: 'numpy.random._generator.Generator' object has no attribute 'randint'

`randint` is one of the older functions, `np.random.randint`, and a `Generator` calls the same thing
`integers`. The older `rand` and `randn` are `random` and `standard_normal` on a `Generator`, which
has none of the old names:

In [16]:
print(rng.integers(1, 7))
for name in ["randint", "rand", "randn", "integers", "random", "standard_normal"]:
    print(f"{name:16} {hasattr(rng, name)}")

3
randint          False
rand             False
randn            False
integers         True
random           True
standard_normal  True


### ValueError: high <= 0

In [17]:
svalbard_warm = np.flatnonzero(readings[2] > 10)
print(svalbard_warm.size)

rng = np.random.default_rng(2025)
rng.integers(svalbard_warm.size)

0


ValueError: high <= 0

Given one argument, `integers` treats it as the upper end, with 0 as the lower, so this asked for a
whole number from 0 up to but not including 0, of which there are none. Svalbard was never above 10
degrees, so there was no warm hour to pick from. Check that a selection holds something before
drawing a position from it.

### No error, and a high value that never appears

In [18]:
rng = np.random.default_rng(5)
rolls = rng.integers(1, 6, size=6000)

print(np.bincount(rolls, minlength=7))

[   0 1221 1151 1218 1213 1197    0]


Six thousand rolls of a die gave not a single 6, because `integers` leaves out `high`. `np.bincount`
counts each value from 0 upward, and both 0 and 6 have a count of 0. A die is `integers(1, 7)`, or
`integers(1, 6, endpoint=True)`:

In [19]:
rng = np.random.default_rng(5)

print(np.bincount(rng.integers(1, 7, size=6000), minlength=7))

[   0 1020  951  995 1019 1015 1000]


### No error, and the same numbers on every pass through a loop

In [20]:
for trial in range(3):
    rng = np.random.default_rng(2025)
    print(rng.integers(1, 7, size=5))

[3 6 6 3 6]
[3 6 6 3 6]
[3 6 6 3 6]


Making the generator inside the loop starts the stream again on every pass, so the three trials are
one trial repeated. Make the generator once, before the loop:

In [21]:
rng = np.random.default_rng(2025)
for trial in range(3):
    print(rng.integers(1, 7, size=5))

[3 6 6 3 6]
[5 4 6 5 6]
[3 1 3 2 3]


### No error, and np.random.seed did not seed the generator

In [22]:
np.random.seed(2025)
first = np.random.default_rng().integers(0, 1_000_000, size=5)

np.random.seed(2025)
second = np.random.default_rng().integers(0, 1_000_000, size=5)

print(np.array_equal(first, second))

False


`np.random.seed` sets the older, hidden generator that functions such as `np.random.randint` use.
`default_rng()` with no seed ignores it and takes a fresh seed from the operating system, so the two
draws differ, and differ again on every run, which is why this cell prints only whether they matched.
Pass the seed to `default_rng` itself.

## Recap

- Make a generator with `np.random.default_rng(seed)`, once, and draw everything from it
- Draw whole numbers with `integers`, which leaves out `high`, and decimals with `random` and `uniform`
- Add noise with `normal`, and sample positions with `choice(..., replace=False)`
- Shuffle and split with `permutation`
- Estimate a share by simulation, and check it against every case when that is possible

## What is next

The **Linear Algebra** notebook multiplies arrays as matrices with the `@` operator, solves systems of
equations, and fits the readings' seasonal and daily swings with least squares.

---

&#8592; **Previous:** [The axis Argument](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/numpy-deep-dive/11-the-axis-argument.ipynb)  &nbsp;·&nbsp;  [NumPy, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/numpy-deep-dive.html)
